# PariShiksha: Retrieval-Ready Study Assistant for NCERT Science

This notebook implements a grounded study assistant for NCERT Class 9 Science content (Chapter 8: Motion).

## Stage 1: Corpus Extraction and Chunking
We extract text from the PDF and split it into manageable chunks with metadata.

In [ ]:
import fitz
import os
from stage1_corpus import extract_text_from_pdf, split_content
from stage2_retrieval import chunk_text, ChunkStore

pdf_path = "data/motion.pdf"
text = extract_text_from_pdf(pdf_path)
sections = split_content(text)

print(f"Extracted {len(text)} characters.")

# Chunking strategy: 500 tokens with 50 token overlap
all_chunks = []
for sec_name, content_list in sections.items():
    for content in content_list:
        chunks = chunk_text(content, {"chapter": "Motion", "section": sec_name})
        all_chunks.extend(chunks)

print(f"Created {len(all_chunks)} chunks.")

## Stage 2: Retrieval System
Implementing BM25 retrieval.

In [ ]:
store = ChunkStore()
store.add_chunks(all_chunks)

query = "What is distance?"
retrieved = store.retrieve(query, k=3)
for i, chunk in enumerate(retrieved):
    print(f"Chunk {i+1} (Source: {chunk['metadata']['section']}):\n{chunk['text'][:200]}...\n")

## Stage 3: Grounded Generation
Using Gemini for answering.

In [ ]:
from stage3_generation import StudyAssistant
import getpass

# Note: In a real environment, set your GOOGLE_API_KEY
api_key = os.getenv("GOOGLE_API_KEY")
assistant = StudyAssistant(api_key)

question = "What is the difference between distance and displacement?"
retrieved_chunks = store.retrieve(question, k=3)
answer = assistant.generate_answer(question, retrieved_chunks)
print(f"Question: {question}\nAnswer: {answer}")